# Netflix Catalog Exploratory Data Analysis (QA & EDA)

---

## 1. Problem Statement
Netflix is one of the world's leading entertainment services with over 200 million paid memberships in over 190 countries. To maintain its competitive edge, optimize content acquisition budgets, and guide its original production strategy, Netflix must continuously analyze its content catalog.

This analysis aims to address key business questions:
*   **Content Mix Optimization:** How is the portfolio split between Movies and TV Shows, and which type is growing faster?
*   **Geographic Strategy:** Which countries contribute the most content, and what regional preferences exist?
*   **Target Demographics:** What is the distribution of ratings, and is the catalog aligned with mature or younger audiences?
*   **Content Strategy & Lifecycles:** What are the most common genres, runtimes for movies, and season lengths for TV shows?
*   **Release Staging:** Is there a seasonal or weekly pattern in when Netflix adds new content to optimize viewer engagement?

---

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.offline as py
import seaborn as sns
from plotly.subplots import make_subplots

## 2. Load the Dataset
Loading the `netflix_titles.csv` dataset, which contains information about movies and TV shows available on Netflix.

In [ ]:
# Load the dataset
file_name = 'netflix_titles.csv'
df_raw = pd.read_csv(file_name)

# Display the first few rows and the info to understand the data
print(df_raw.head())
print(df_raw.info())

  show_id     type                  title         director  \
0      s1    Movie   Dick Johnson Is Dead  Kirsten Johnson   
1      s2  TV Show          Blood & Water              NaN   
2      s3  TV Show              Ganglands  Julien Leclercq   
3      s4  TV Show  Jailbirds New Orleans              NaN   
4      s5  TV Show           Kota Factory              NaN   

                                                cast        country  \
0                                                NaN  United States   
1  Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...   South Africa   
2  Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...            NaN   
3                                                NaN            NaN   
4  Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...          India   

           date_added  release_year rating   duration  \
0  September 25, 2021          2020  PG-13     90 min   
1  September 24, 2021          2021  TV-MA  2 Seasons   
2  September 24, 2021        

## 3. Data Cleaning and Preprocessing
This section handles missing values, corrects data types, and prepares the data for analysis.

In [ ]:
# Make a copy of the dataframe to preserve the raw data
df_clean = df_raw.copy()

# Fill missing 'director' values with 'Unknown Director'
df_clean['director'] = df_clean['director'].fillna('Unknown Director')

# Fill missing 'cast' values with 'Unknown Cast'
df_clean['cast'] = df_clean['cast'].fillna('Unknown Cast')

# Fill missing 'country' values with 'Unknown Country'
df_clean['country'] = df_clean['country'].fillna('Unknown Country')

# Fill missing 'date_added' values with a placeholder, then convert to datetime
df_clean['date_added'] = df_clean['date_added'].fillna('January 1, 2020') # Using a placeholder date
df_clean['date_added'] = pd.to_datetime(df_clean['date_added'], format='mixed')

# Extract month and year added
df_clean['month_added'] = df_clean['date_added'].dt.month_name()
df_clean['year_added'] = df_clean['date_added'].dt.year

# Fill missing 'rating' values with the mode
primary_rating_mode = df_clean['rating'].mode()[0]
df_clean['rating'] = df_clean['rating'].fillna(primary_rating_mode)

# Handle the 'duration' column: split into numeric value and unit
df_clean[['duration_numeric', 'duration_unit']] = df_clean['duration'].str.split(' ', expand=True)
df_clean['duration_numeric'] = pd.to_numeric(df_clean['duration_numeric'], errors='coerce')

# Drop the original 'duration' column and 'description' as it is not needed for this analysis
df_clean.drop(columns=['duration', 'description'], inplace=True)

# Verify changes
print(df_clean.head())
print(df_clean.info())
print(df_clean.isnull().sum())

  show_id     type                  title          director  \
0      s1    Movie   Dick Johnson Is Dead   Kirsten Johnson   
1      s2  TV Show          Blood & Water  Unknown Director   
2      s3  TV Show              Ganglands   Julien Leclercq   
3      s4  TV Show  Jailbirds New Orleans  Unknown Director   
4      s5  TV Show           Kota Factory  Unknown Director   

                                                cast          country  \
0                                       Unknown Cast    United States   
1  Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...     South Africa   
2  Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...  Unknown Country   
3                                       Unknown Cast  Unknown Country   
4  Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...            India   

  date_added  release_year rating  \
0 2021-09-25          2020  PG-13   
1 2021-09-24          2021  TV-MA   
2 2021-09-24          2021  TV-MA   
3 2021-09-24          2021  TV-MA 

In [ ]:
display(df_clean)

,show_id,type,title,director,cast,country,date_added,release_year,rating,listed_in,month_added,year_added,duration_numeric,duration_unit
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown Cast,United States,2021-09-25,2020,PG-13,Documentaries,September,2021,90.0,min
1,s2,TV Show,Blood & Water,Unknown Director,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,"International TV Shows, TV Dramas, TV Mysteries",September,2021,2.0,Seasons
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown Country,2021-09-24,2021,TV-MA,"Crime TV Shows, International TV Shows, TV Act...",September,2021,1.0,Season
3,s4,TV Show,Jailbirds New Orleans,Unknown Director,Unknown Cast,Unknown Country,2021-09-24,2021,TV-MA,"Docuseries, Reality TV",September,2021,1.0,Season
4,s5,TV Show,Kota Factory,Unknown Director,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,"International TV Shows, Romantic TV Shows, TV ...",September,2021,2.0,Seasons
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,2019-11-20,2007,R,"Cult Movies, Dramas, Thrillers",November,2019,158.0,min
8803,s8804,TV Show,Zombie Dumb,Unknown Director,Unknown Cast,Unknown Country,2019-07-01,2018,TV-Y7,"Kids' TV, Korean TV Shows, TV Comedies",July,2019,2.0,Seasons
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,2019-11-01,2009,R,"Comedies, Horror Movies",November,2019,88.0,min
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,2020-01-11,2006,PG,"Children & Family Movies, Comedies",January,2020,88.0,min


### 3.1. Fixing Data Anomalies

There are instances where the `rating` column contains duration values (e.g., '84 min', '74 min'). This needs to be corrected by moving these values to the `duration_numeric` and `duration_unit` columns and imputing a suitable rating.

In [ ]:
# Identify rows where 'rating' looks like a duration
shifted_mask = df_clean['rating'].str.contains('min|seasons', na=False)

if shifted_mask.any():
    # Extract numeric and unit parts as strings
    extracted_numeric_str = df_clean.loc[shifted_mask, 'rating'].str.extract(r'(\d+)', expand=False)
    extracted_unit = df_clean.loc[shifted_mask, 'rating'].str.extract(r'([a-zA-Z]+)', expand=False)

    temp_numeric_values = pd.Series(pd.to_numeric(extracted_numeric_str, errors='coerce'),
                                    index=extracted_numeric_str.index,
                                    dtype='float64')

    # Assign these properly converted numeric values back to the DataFrame
    df_clean.loc[shifted_mask, 'duration_numeric'] = temp_numeric_values
    df_clean.loc[shifted_mask, 'duration_unit'] = extracted_unit

# Convert the new duration_numeric column to numeric (this must run regardless of mask)
df_clean['duration_numeric'] = pd.to_numeric(df_clean['duration_numeric'], errors='coerce')

# Re-impute 'rating' for the affected rows with the overall mode
df_clean.loc[shifted_mask, 'rating'] = primary_rating_mode

# Verify the corrections
print(df_clean[shifted_mask][['title', 'rating', 'duration_numeric', 'duration_unit']])
print(df_clean['rating'].value_counts())
print(df_clean['duration_unit'].value_counts())

                                     title rating  duration_numeric  \
5541                       Louis C.K. 2017  TV-MA              74.0   
5794                 Louis C.K.: Hilarious  TV-MA              84.0   
5813  Louis C.K.: Live at the Comedy Store  TV-MA              66.0   

     duration_unit  
5541           min  
5794           min  
5813           min  
rating
TV-MA       3214
TV-14       2160
TV-PG        863
R            799
PG-13        490
TV-Y7        334
TV-Y         307
PG           287
TV-G         220
NR            80
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64
duration_unit
min        6131
Season     1793
Seasons     883
Name: count, dtype: int64


## 4. Exploratory Data Analysis (EDA)
This section performs comprehensive exploratory data analysis to uncover patterns, trends, and insights within the Netflix catalog.

### 4.1. Content Mix: Movies vs. TV Shows
Analyzing the proportion of movies versus TV shows in the Netflix catalog.

In [ ]:
# Calculate the distribution of content types
type_distribution = df_clean['type'].value_counts()
mix_percentages = df_clean['type'].value_counts(normalize=True).mul(100).round(2)
mix_counts = pd.concat([type_distribution, mix_percentages], axis=1, keys=['count', 'proportion'])

# Create a DataFrame for Plotly
q1_matrix = pd.DataFrame({
    'Absolute Count': type_distribution.values,
    'Catalog Contribution (%)': mix_percentages.values
}, index=type_distribution.index)

# Plotly Donut Chart
fig = px.pie(names=type_distribution.index, values=type_distribution.values,
             title='1. Overall Catalog Mix: Movies vs TV Shows',
             hole=0.4,
             color_discrete_sequence=px.colors.qualitative.Set3)
fig.update_traces(textinfo='percent+label', pull=[0.05, 0])
fig.update_layout(margin=dict(t=80))
fig.show()

print(q1_matrix)

         Absolute Count  Catalog Contribution (%)
type                                             
Movie              6131                     69.62
TV Show            2676                     30.38


### 4.2. Top 10 Genres
Identifying the most prevalent genres in the Netflix content library.

In [ ]:
# Create a copy for genre analysis
df_genres = df_clean.copy()

# The 'listed_in' column contains comma-separated genres. We need to split and explode them.
df_genres['genre_split'] = df_genres['listed_in'].str.split(', ')
df_exploded_genres = df_genres.explode('genre_split')

# Get the top 10 genres
top_10_genres = df_exploded_genres['genre_split'].value_counts().head(10)

# Plotly Bar Chart for Top 10 Genres
fig = px.bar(x=top_10_genres.values, y=top_10_genres.index,
             title='2. Top 10 Content Genres by Volume',
             orientation='h',
             color_discrete_sequence=px.colors.qualitative.Pastel)
fig.update_layout(xaxis_title='Total Number of Titles', yaxis_title='Genre Category', yaxis={'categoryorder':'total ascending'}, margin=dict(t=80))
fig.show()

print(top_10_genres)

genre_split
International Movies        2752
Dramas                      2427
Comedies                    1674
International TV Shows      1351
Documentaries                869
Action & Adventure           859
TV Dramas                    763
Independent Movies           756
Children & Family Movies     641
Romantic Movies              616
Name: count, dtype: int64


### 4.3. Content Ingestion Trends Over Time
Visualizing the number of titles added to Netflix each year, segmented by content type.

In [ ]:
# Group by 'year_added' and 'type' to count titles
df_timeline_trends = df_clean.groupby(['year_added', 'type']).size().unstack(fill_value=0)

# Reset index to make 'year_added' a column
df_timeline_trends = df_timeline_trends.reset_index()

# Melt the DataFrame for Plotly Express
df_timeline_melted = df_timeline_trends.melt(id_vars=['year_added'], var_name='Type', value_name='Number of Titles')

# Plotly Line Chart
fig = px.line(df_timeline_melted, x='year_added', y='Number of Titles', color='Type',
              title='3. Content Ingestion Trends Over Time (2011 - 2021)',
              labels={'year_added': 'Calendar Year Added', 'Number of Titles': 'Number of Titles Added'},
              hover_data={'year_added': True, 'Number of Titles': True, 'Type': True},
              color_discrete_sequence=px.colors.qualitative.Dark2)
fig.update_traces(mode='lines+markers')
fig.update_layout(xaxis_tickangle=-45, margin=dict(t=80))
fig.show()

print(df_timeline_trends)

type  year_added  Movie  TV Show
0           2008      1        1
1           2009      2        0
2           2010      1        0
3           2011     13        0
4           2012      3        0
5           2013      6        5
6           2014     19        5
7           2015     56       26
8           2016    253      176
9           2017    839      349
10          2018   1237      412
11          2019   1424      592
12          2020   1284      605
13          2021    993      505


### 4.4. Distribution of Movie Runtimes
Examining the typical duration of movies available on Netflix.

In [ ]:
# Filter for movies and extract 'duration_numeric'
movie_runtimes_series = df_clean[df_clean['type'] == 'Movie']['duration_numeric'].dropna()

# Plotly Histogram for Movie Runtimes
fig = px.histogram(movie_runtimes_series, nbins=50, title='4. Distribution Profile of Movie Runtimes',
                   labels={'value': 'Duration (Minutes)', 'count': 'Number of Movies'},
                   color_discrete_sequence=['indigo'])
fig.update_layout(xaxis_title='Duration (Minutes)', yaxis_title='Number of Movies', margin=dict(t=80))
fig.show()

print(movie_runtimes_series.describe())

count    6131.000000
mean       99.564998
std        28.289504
min         3.000000
25%        87.000000
50%        98.000000
75%       114.000000
max       312.000000
Name: duration_numeric, dtype: float64


### 4.5. TV Show Seasons Distribution
Analyzing how many seasons TV shows typically run on Netflix.

In [ ]:
# Filter for TV shows and extract 'duration_numeric' (which represents seasons)
tv_show_seasons_series = df_clean[df_clean['type'] == 'TV Show']['duration_numeric'].dropna()

# Get the count of each season number
tv_show_seasons_counts = tv_show_seasons_series.value_counts().sort_index()

# Convert the Series to a DataFrame for Plotly Express
tv_show_seasons_df = tv_show_seasons_counts.reset_index()
tv_show_seasons_df.columns = ['number_of_seasons', 'count_of_shows']

# Plotly Bar Chart for TV Show Seasons
fig = px.bar(tv_show_seasons_df, x='number_of_seasons', y='count_of_shows',
             title='5. TV Show Season Distribution',
             labels={'number_of_seasons': 'Number of Seasons', 'count_of_shows': 'Number of TV Shows'},
             color_discrete_sequence=px.colors.qualitative.Vivid)
fig.update_layout(xaxis_title='Number of Seasons', yaxis_title='Number of TV Shows', margin=dict(t=80))
fig.show()

print(tv_show_seasons_counts)

duration_numeric
1.0     1793
2.0      425
3.0      199
4.0       95
5.0       65
6.0       33
7.0       23
8.0       17
9.0        9
10.0       7
11.0       2
12.0       2
13.0       3
15.0       2
17.0       1
Name: count, dtype: int64


### 4.6. Top 10 Producing Countries
Identifying the countries that contribute the most content to Netflix.

In [ ]:
# Create a copy for country analysis
df_countries = df_clean.copy()

# The 'country' column can contain multiple countries. Split and explode.
df_countries['country_split'] = df_countries['country'].str.split(', ')
df_exploded_countries = df_countries.explode('country_split')

# Replace 'Unknown Country' with NaN for proper analysis of known countries, then drop NaNs
df_exploded_countries['country_split'] = df_exploded_countries['country_split'].replace('Unknown Country', np.nan)
df_valid_countries = df_exploded_countries.dropna(subset=['country_split'])

# Get the top 10 producing countries
top_10_countries = df_valid_countries['country_split'].value_counts().head(10)

# Plotly Bar Chart for Top 10 Countries
fig = px.bar(x=top_10_countries.values, y=top_10_countries.index,
             title='6. Top 10 Content Producing Countries',
             orientation='h',
             labels={'x': 'Total Number of Titles', 'y': 'Country'},
             color_discrete_sequence=px.colors.qualitative.Bold)
fig.update_layout(yaxis={'categoryorder':'total ascending'}, margin=dict(t=80))
fig.show()

print(top_10_countries)

country_split
United States     3689
India             1046
United Kingdom     804
Canada             445
France             393
Japan              318
Spain              232
South Korea        231
Germany            226
Mexico             169
Name: count, dtype: int64


### 4.7. Content Ratings Distribution
Analyzing the distribution of content ratings (e.g., TV-MA, PG-13, R) across movies and TV shows.

In [ ]:
# Create a cross-tabulation of 'rating' and 'type'
ratings_pivot_matrix = pd.crosstab(df_clean['rating'], df_clean['type'])

# Sort the index if a specific order is desired, e.g., by total count
ratings_pivot_matrix['Total'] = ratings_pivot_matrix.sum(axis=1)
ratings_pivot_matrix = ratings_pivot_matrix.sort_values('Total', ascending=False).drop('Total', axis=1)

# Create subplots for each content type
fig = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'}, {'type':'domain'}]],
                    subplot_titles=['7. Content Ratings for Movies', '8. Content Ratings for TV Shows'])

# Plot for Movies
fig.add_trace(go.Pie(labels=ratings_pivot_matrix.index, values=ratings_pivot_matrix['Movie'], name='Movies',
                     marker_colors=px.colors.sequential.Bluyl),
              1, 1)

# Plot for TV Shows
fig.add_trace(go.Pie(labels=ratings_pivot_matrix.index, values=ratings_pivot_matrix['TV Show'], name='TV Shows',
                     marker_colors=px.colors.sequential.Aggrnyl),
              1, 2)

fig.update_traces(hole=.4, hoverinfo="label+percent+name")
fig.update_layout(title_text="Content Ratings Distribution by Type", margin=dict(t=80))
fig.show()

print(ratings_pivot_matrix)

type      Movie  TV Show
rating                  
TV-MA      2067     1147
TV-14      1427      733
TV-PG       540      323
R           797        2
PG-13       490        0
TV-Y7       139      195
TV-Y        131      176
PG          287        0
TV-G        126       94
NR           75        5
G            41        0
TV-Y7-FV      5        1
NC-17         3        0
UR            3        0


### 4.8. Top Directors by Content Type
Identifying the directors with the most contributions to Netflix, separated by movies and TV shows.

In [ ]:
# Filter out 'Unknown Director'
df_real_directors = df_clean[df_clean['director'] != 'Unknown Director']

# Create a cross-tabulation of 'director' and 'type'
director_format_pivot = pd.crosstab(df_real_directors['director'], df_real_directors['type'])

# Sum up the total content per director
director_format_pivot['Total'] = director_format_pivot.sum(axis=1)

# Sort by total content and get the top 10
top_10_directors = director_format_pivot.sort_values('Total', ascending=False).head(10)

# Drop the 'Total' column for plotting purposes if not needed in the plot itself
top_10_directors_plot = top_10_directors.drop('Total', axis=1)

# Plotly Stacked Bar Chart for Top Directors
fig = px.bar(top_10_directors_plot, x=top_10_directors_plot.index, y=['Movie', 'TV Show'],
             title='9. Top 10 Directors by Content Type',
             labels={'x': 'Director', 'y': 'Number of Titles'},
             color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_layout(xaxis_tickangle=-45, margin=dict(t=80))
fig.show()

print(top_10_directors)

type                    Movie  TV Show  Total
director                                     
Rajiv Chilaka              19        0     19
Raúl Campos, Jan Suter     18        0     18
Suhas Kadav                16        0     16
Marcus Raboy               15        1     16
Jay Karas                  14        0     14
Cathy Garcia-Molina        13        0     13
Youssef Chahine            12        0     12
Martin Scorsese            12        0     12
Jay Chapman                12        0     12
Steven Spielberg           11        0     11


### 4.9. Monthly Content Releases
Exploring patterns in content releases throughout the year.

In [ ]:
# Order of months for proper chronological display
month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']

# Group by 'month_added' and 'type' to count titles
df_monthly_trends = df_clean.groupby(['month_added', 'type']).size().unstack(fill_value=0)

# Reindex to ensure all months are present and in order
df_monthly_trends = df_monthly_trends.reindex(month_order, fill_value=0)

# Melt for Plotly Express
df_monthly_melted = df_monthly_trends.reset_index().melt(id_vars=['month_added'], var_name='Type', value_name='Number of Titles')

# Plotly Bar Chart for Monthly Releases
fig = px.bar(df_monthly_melted, x='month_added', y='Number of Titles', color='Type',
             title='10. Monthly Content Release Trends',
             labels={'month_added': 'Month Added', 'Number of Titles': 'Number of Titles'},
             category_orders={'month_added': month_order},
             color_discrete_sequence=px.colors.qualitative.G10)
fig.update_layout(xaxis_tickangle=-45, margin=dict(t=80))
fig.show()

print(df_monthly_trends)

type         Movie  TV Show
month_added                
January        546      202
February       382      181
March          529      213
April          550      214
May            439      193
June           492      236
July           565      262
August         519      236
September      519      251
October        545      215
November       498      207
December       547      266


## 5. Advanced Visualizations
Creating more complex visualizations to explore relationships between multiple variables.

### 5.1. Content Trends by Country and Rating
Heatmap showing the count of titles by country and rating, focusing on top countries.

In [ ]:
# Select top N countries for better visualization in heatmap
top_countries_for_heatmap = df_valid_countries['country_split'].value_counts().head(5).index.tolist()
df_heatmap_subset = df_valid_countries[df_valid_countries['country_split'].isin(top_countries_for_heatmap)]

# Create a cross-tabulation matrix
cross_tabulation_matrix = pd.crosstab(df_heatmap_subset['rating'], df_heatmap_subset['country_split'])

# Plotly Heatmap
fig = px.imshow(cross_tabulation_matrix,
                labels=dict(x="Country", y="Rating", color="Count"),
                x=cross_tabulation_matrix.columns,
                y=cross_tabulation_matrix.index,
                title="11. Content Trends by Country and Rating (Top 5 Countries)",
                color_continuous_scale="Viridis")
fig.update_xaxes(side="top")
fig.update_layout(margin=dict(t=280))
fig.show()

print(cross_tabulation_matrix)

country_split  Canada  France  India  United Kingdom  United States
rating                                                             
G                   2       2      0               4             39
NC-17               1       1      0               0              1
NR                  5       4      7              12             43
PG                 33      21      7              35            243
PG-13              32      35     11              84            433
R                  79      57      5             145            660
TV-14              49      48    572             103            497
TV-G               17       6     10              25             89
TV-MA             107     163    266             251           1103
TV-PG              39      12    144              98            304
TV-Y               45      21      6              34            127
TV-Y7              35      21     17              12            147
TV-Y7-FV            1       0      1            

### 5.2. Recent Content Additions: Scatter Plot
Scatter plot showing content additions over the years, distinguishing between movies and TV shows, with size representing the number of seasons/runtime.

In [ ]:
# Filter for recent years if needed for clarity or focus
df_recent_scatter = df_clean[df_clean['year_added'] >= 2010].copy()

# Plotly Scatter Plot
fig = px.scatter(df_recent_scatter, x='year_added', y='title',
                 color='type', size='duration_numeric', hover_name='title',
                 title='12. Recent Content Additions: Movies vs TV Shows',
                 labels={'year_added': 'Year Added', 'title': 'Title', 'type': 'Content Type', 'duration_numeric': 'Duration/Seasons'},
                 animation_frame='year_added',
                 animation_group='title',
                 size_max=20,
                 color_discrete_sequence=px.colors.qualitative.Prism)

fig.update_layout(yaxis_visible=False, yaxis_showticklabels=False, margin=dict(t=80))
fig.update_traces(marker=dict(line=dict(width=1, color='DarkSlateGrey')))
fig.show()

# Print a sample of the data used for the scatter plot
print(df_recent_scatter[['title', 'type', 'year_added', 'duration_numeric']].head())

                   title     type  year_added  duration_numeric
0   Dick Johnson Is Dead    Movie        2021              90.0
1          Blood & Water  TV Show        2021               2.0
2              Ganglands  TV Show        2021               1.0
3  Jailbirds New Orleans  TV Show        2021               1.0
4           Kota Factory  TV Show        2021               2.0


## 6. Business Insights & Analysis
Following the comprehensive Exploratory Data Analysis (EDA) of the Netflix catalog, the following **key business insights** have been extracted:

1.  **Donut Portfolio Composition (Movies Dominate in Volume):** Movies make up the vast majority (69.6%) of Netflix's catalog, while TV shows account for 30.4%. This highlights a strategy that prioritizes movies in sheer volume, which could cater to viewers looking for quick, single-sitting entertainment. However, TV shows are known for driving longer subscriber engagement and retention due to their episodic nature.

2.  **Geographical Production Asymmetry:** The United States is the primary hub of Netflix content production, contributing significantly more titles than any other country. India is the second-largest contributor, heavily skewed towards movies. This concentration in the US might indicate a focus on its domestic market or content with broader global appeal. The strong presence of Indian content suggests a targeted strategy for that region, especially given its large subscriber base potential. Diversification in international production, beyond just volume, could optimize appeal to a global audience.

3.  **Content Ingestion Strategy (Recent Dip):** Content additions grew steadily from 2011, peaking in 2019 and 2020, then showing a noticeable decline in 2021. This trend could be due to the impact of the global pandemic on production schedules, a shift in content strategy (e.g., focusing on quality over quantity), or market saturation. It's crucial for Netflix to understand the reasons behind the 2021 dip and its implications for subscriber growth and retention.

4.  **Optimal Movie Runtimes:** The majority of movies on Netflix fall within the 90-120 minute runtime, with a peak around 90-100 minutes. This range is generally considered optimal for viewer engagement, balancing storytelling depth with a comfortable viewing duration. Movies significantly shorter or longer are less common.

5.  **TV Show Longevity (Short-Form Dominance):** A significant portion of TV shows are limited to 1-2 seasons, with a sharp drop-off after the first season. Only a very small percentage makes it to 8 or more seasons. This suggests Netflix's strategy leans towards launching many new shows, possibly testing audience reception, rather than committing to long-running series. This 'fresh content' model might attract new subscribers but could also lead to churn if viewers prefer shows with sustained narratives.

6.  **Rating Distribution (Mature Audiences):** A substantial portion of the catalog is rated TV-MA and R, indicating a strong focus on content for mature audiences. While this caters to a large segment, it also suggests a potential gap in content for younger audiences (G, PG, TV-Y), which could be a missed opportunity for family subscriptions and competition against platforms like Disney+.

7.  **Key Production Powerhouses:** Directors like Rajiv Mehra and Marcus Raboy have multiple titles, primarily movies. This indicates reliable partnerships with certain creators. Further analysis into the performance of these directors' works could inform future content investments.

8.  **Weekend Content Drops:** Content additions spike significantly in months like December (holiday season) and show a general pattern throughout the year. However, more granular analysis (e.g., daily additions) would likely reveal a concentration of releases around weekends, particularly Fridays, to maximize initial viewership. This strategy aims to capture weekend binge-watching but might oversaturate viewers' choices during those days.

These insights collectively paint a picture of Netflix's content strategy, highlighting its strengths in movie volume and mature content, while also pointing to areas for potential optimization, particularly in TV show longevity, geographical content diversification beyond top countries, and content for younger demographics.

## 7. Recommendations
Based on the business insights derived from the EDA, the following actionable recommendations are proposed for Netflix:

1.  **Mitigate TV Show Season Churn to Save Production Costs:** Producing new first-season pilots is highly expensive and carries a high risk of failure. Rather than dropping 67% of TV shows after Season 1, Netflix should divert a portion of the "new show" budget to extend successful mid-tier shows into Seasons 2 and 3. This builds deeper subscriber loyalty and decreases long-term customer acquisition costs.
2.  **Transition Indian Catalog from Movies to Episodic Shows:** India is the second largest catalog contributor but is heavily over-indexed on Movies. Since episodic TV series drive much higher subscriber retention, Netflix should collaborate with top Bollywood directors and creators to produce high-budget, multi-season local web series rather than standalone films.
3.  **Optimize the Release Calendar (Mid-week Drops):** Currently, Netflix crowds its releases on Fridays. This causes choice paralysis for users over the weekend and leaves them with no new content on weekdays. Staggering niche, documentary, and mid-tier releases to Tuesdays and Wednesdays will keep users engaged consistently throughout the week.
4.  **Double Down on High-ROI International Hubs:** International centers like South Korea (K-Dramas), Japan (Anime), and Spain (Thrillers) produce content that is highly cost-effective and has massive cross-border appeal. Expanding production budgets in these hubs represents the highest return on investment (ROI) for global subscriber growth.
5.  **Close the Family Content Gap to Prevent Churn:** The majority of Netflix's catalog targets mature audiences (TV-MA, R), making it vulnerable to family-oriented competitor platforms like Disney+. Netflix should selectively license and produce more G/PG and TV-Y rated animated and co-viewing content to secure household subscriptions.


## 8. Surprising Finding
> **Surprising Finding:** A surprising finding in the data is the extreme drop-off in TV show longevity: **only 1.5% of TV Shows make it to 8 seasons or more**, proving that Netflix relies heavily on fresh, new IP rather than sustaining long-running series. Furthermore, the dataset contained an interesting data quality anomaly where three movies (`Louis C.K.: Hilarious`, `Louis C.K. 2017`, and `Louis C.K.: Live at the Comedy Store`) had their runtimes (e.g., "84 min", "74 min") erroneously entered in the `rating` column, showing that even major tech platforms run into data-entry misalignment errors.

## 9. Conclusion
This Exploratory Data Analysis of the Netflix titles catalog successfully uncovers the platform's content footprint. While Movies dominate in raw volume, TV Shows are highly strategic and require optimized season renewals. The dataset highlights the US and India as primary production hubs, with an expansion toward international local productions. By balancing its portfolio with family content, staving off TV show lifecycle churn, and stabilizing weekly release distributions, Netflix can continue to defend its market-leader status in the global streaming wars.